In [ ]:
# ==============================================================================
# CELL 1: IMPORTS & CONFIGURATION
# ==============================================================================
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import AdamW 
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, matthews_corrcoef, roc_auc_score, 
    average_precision_score, confusion_matrix, roc_curve, f1_score
)
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# --- CONFIGURATION ---
class Config:
    # Files (Update paths if necessary)
    POSITIVE_CSV = "datasets/fusion_gene_positive_bp_information_with_class_for_modeling.txt"
    NEGATIVE_CSV = "datasets/fusion_gene_negative_bp_information_with_class_for_modeling.txt"
    TEST_CSV = "datasets/fusion_gene_positive_bp_information_with_class_for_testing.txt"
    
    # Extra FASTA Data
    EXTRA_POS_FASTA = "datasets/blast_validated_chimeras.fasta,datasets/cosmic_high_confidence_sequences.fna,datasets/chimeras_43466.fa"
    NEG_FASTA_CANONICAL = "datasets/false_negative_candidates.fasta"
    NEG_FASTA_SYNTHETIC = "datasets/false_positive_candidates.fasta"
    
    # Model & Training
    OUTPUT_DIR = "./hyenadna_v9.7_checkpoints"
    MODEL_NAME = "LongSafari/hyenadna-small-32k-seqlen-hf"
    MAX_LEN = 20480  # 20kb Context
    BATCH_SIZE = 8
    EPOCHS = 3
    LEARNING_RATE = 1e-5
    VAL_SPLIT = 0.2
    SEED = 42
    
    # Logic: High Confidence Threshold for Breakpoint Plotting
    CONFIDENCE_THRESHOLD = 0.90 

if not os.path.exists(Config.OUTPUT_DIR):
    os.makedirs(Config.OUTPUT_DIR)

# --- REPRODUCIBILITY ---
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(Config.SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Setup Complete. Device: {device}")

In [ ]:
# ==============================================================================
# CELL 2: CLASS DEFINITIONS (Fixed Data Leakage)
# ==============================================================================

# --- DATA ENGINEERING ---
class DataPreparator:
    COLUMNS = ["Hgene","Hchr","Hbp","Hstrand","Tgene","Tchr","Tbp","Tstrand","5'-gene sequence (10Kb)","3'-gene sequence (10Kb)"]

    def __init__(self, config):
        self.cfg = config
        self.trans_table = str.maketrans("ATCGN", "TAGCN")
        # NEW: Track sequences used in training to prevent leakage
        self.train_sequences = set()

    def _trim_artifacts(self, sequence: str) -> str:
        if not isinstance(sequence, str): return ""
        sequence = sequence.upper().rstrip('N')
        if sequence.endswith('AAAAAAAAAA'):
            for i in range(len(sequence)-1, -1, -1):
                if sequence[i] != 'A': return sequence[:i+1]
        return sequence

    def _get_reverse_complement(self, sequence: str) -> str:
        return sequence.upper().translate(self.trans_table)[::-1]

    def _apply_jitter(self, seq_5p: str, seq_3p: str) -> str:
        seq_5p = self._trim_artifacts(seq_5p)
        seq_3p = self._trim_artifacts(seq_3p)
        shift = random.randint(-2500, 2500)
        len_5p_take = 10240 + shift
        len_3p_take = 10240 - shift
        
        def process(s, le, from_start=False):
            if len(s) >= le: return s[:le] if from_start else s[-le:]
            else: return (s + 'N'*(le-len(s))) if from_start else ('N'*(le-len(s)) + s)

        part1 = process(seq_5p, len_5p_take, from_start=False)
        part2 = process(seq_3p, len_3p_take, from_start=True)
        return part1 + part2

    def _load_fasta(self, path: str) -> list:
        if not os.path.exists(path): return []
        seqs, curr = [], []
        with open(path, 'r') as f:
            for line in f:
                line = line.strip()
                if line.startswith('>'):
                    if curr: seqs.append(self._trim_artifacts("".join(curr)))
                    curr = []
                else: curr.append(line)
            if curr: seqs.append(self._trim_artifacts("".join(curr)))
        return seqs

    def load_training_data(self):
        print("\n--- 1. LOADING TRAINING DATA ---")
        # Positives
        if os.path.exists(self.cfg.POSITIVE_CSV):
            pos_df = pd.read_csv(self.cfg.POSITIVE_CSV, header=None, names=self.COLUMNS, sep='\t')
            pos_df['sequence'] = pos_df.apply(lambda r: self._apply_jitter(r["5'-gene sequence (10Kb)"], r["3'-gene sequence (10Kb)"]), axis=1)
            pos_df['label'] = 1
        else:
            print("⚠️ POSITIVE CSV missing! Using dummy data.")
            pos_df = pd.DataFrame({'sequence': ['ACGT'*500]*100, 'label': [1]*100})

        # Extra FASTA
        extra_seqs = []
        for p in self.cfg.EXTRA_POS_FASTA.split(','):
            extra_seqs.extend(self._load_fasta(p.strip()))
        if extra_seqs:
            pos_df = pd.concat([pos_df[['sequence', 'label']], pd.DataFrame({'sequence': extra_seqs, 'label': 1})])

        # Negatives
        neg_seqs = []
        if os.path.exists(self.cfg.NEGATIVE_CSV):
            neg_df = pd.read_csv(self.cfg.NEGATIVE_CSV, header=None, names=self.COLUMNS, sep='\t')
            neg_seqs.extend(neg_df.apply(lambda r: self._apply_jitter(r["5'-gene sequence (10Kb)"], r["3'-gene sequence (10Kb)"]), axis=1).tolist())
        
        neg_seqs.extend(self._load_fasta(self.cfg.NEG_FASTA_CANONICAL))
        neg_seqs.extend(self._load_fasta(self.cfg.NEG_FASTA_SYNTHETIC))
        if not neg_seqs: neg_seqs = ['TGCA'*500]*100
        neg_df = pd.DataFrame({'sequence': neg_seqs, 'label': 0})
        
        # Augmentation
        print("  > Augmenting Negatives (Reverse Complement)...")
        aug_df = neg_df.copy()
        aug_df['sequence'] = aug_df['sequence'].apply(self._get_reverse_complement)
        neg_df = pd.concat([neg_df, aug_df], ignore_index=True)

        combined = pd.concat([pos_df[['sequence', 'label']], neg_df[['sequence', 'label']]], ignore_index=True)
        combined.drop_duplicates(subset=['sequence'], inplace=True)
        combined = combined.sample(frac=1, random_state=Config.SEED).reset_index(drop=True)
        
        # NEW: Store training sequences to prevent leakage
        self.train_sequences = set(combined['sequence'].unique())
        print(f"  > Indexed {len(self.train_sequences)} unique training sequences for leakage prevention.")
        
        print(f"✅ TRAIN/VAL READY: {len(combined)} samples (Pos: {sum(combined['label']==1)}, Neg: {sum(combined['label']==0)})")
        return combined

    def load_test_set(self):
        print("\n--- 2. LOADING BLIND TEST SET ---")
        if not os.path.exists(self.cfg.TEST_CSV):
            print("⚠️ TEST CSV not found! Using dummy test set.")
            return pd.DataFrame({'sequence': ['ACGT'*500]*50, 'label': [1]*50})
        
        # 1. Load Positives (Label 1)
        test_pos = pd.read_csv(self.cfg.TEST_CSV, header=None, names=self.COLUMNS, sep='\t')
        test_pos['sequence'] = test_pos.apply(lambda r: self._apply_jitter(r["5'-gene sequence (10Kb)"], r["3'-gene sequence (10Kb)"]), axis=1)
        # Ensure Test Positives are not in training (unlikely, but good safety)
        test_pos = test_pos[~test_pos['sequence'].isin(self.train_sequences)]
        test_pos['label'] = 1
        n_pos = len(test_pos)
        print(f"  > Loaded {n_pos} Unique Test Positives.")
        
        # 2. Generate Balanced Negatives (Label 0)
        # Load Pool
        raw_negs = self._load_fasta(self.cfg.NEG_FASTA_SYNTHETIC)
        
        # NEW: Filter out ANY sequence used in training
        clean_negs = [s for s in raw_negs if s not in self.train_sequences]
        
        # Also check Reverse Complements (if model learned RC in training, standard in test is leakage)
        # (Optional but strict)
        clean_negs = [s for s in clean_negs if self._get_reverse_complement(s) not in self.train_sequences]
        
        print(f"  > Pool after removing training data: {len(clean_negs)} (Original: {len(raw_negs)})")
        
        if len(clean_negs) < n_pos:
            print("  ⚠️ WARNING: Running low on unique negatives! Generating RCs...")
            clean_negs.extend([self._get_reverse_complement(s) for s in clean_negs])
            # Filter again just in case
            clean_negs = [s for s in clean_negs if s not in self.train_sequences]
            
        # Sample
        if len(clean_negs) > 0:
            selected_negs = random.sample(clean_negs, min(len(clean_negs), n_pos))
            # If we simply don't have enough, we duplicate (better than leakage)
            while len(selected_negs) < n_pos:
                selected_negs.append(selected_negs[random.randint(0, len(selected_negs)-1)])
        else:
            print("  ❌ CRITICAL: No unique negatives left! Creating dummy negatives to allow run.")
            selected_negs = ['TGCA' * 200] * n_pos
            
        test_neg = pd.DataFrame({'sequence': selected_negs, 'label': 0})
        
        # 3. Combine
        test_combined = pd.concat([test_pos[['sequence', 'label']], test_neg[['sequence', 'label']]], ignore_index=True)
        test_combined = test_combined.sample(frac=1, random_state=Config.SEED).reset_index(drop=True)
        
        print(f"✅ TEST SET READY: {len(test_combined)} samples (Balanced 50/50)")
        return test_combined

# --- DATASET & MODEL (Kept same) ---
class DNABreakpointDataset(Dataset):
    def __init__(self, sequences, labels, tokenizer, max_len):
        self.sequences = sequences
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self): return len(self.sequences)
    def __getitem__(self, idx):
        seq = str(self.sequences[idx])
        label = self.labels[idx]
        enc = self.tokenizer(seq, truncation=True, max_length=self.max_len, padding='max_length', return_tensors='pt')
        input_ids = enc['input_ids'].squeeze(0)
        mask = enc['attention_mask'].squeeze(0) if 'attention_mask' in enc else (input_ids != (self.tokenizer.pad_token_id or 0)).long()
        return {'input_ids': input_ids, 'attention_mask': mask, 'labels': torch.tensor(label, dtype=torch.long)}

class HyenaDNAClassifier(nn.Module):
    def __init__(self, model_name, num_labels=2):
        super().__init__()
        self.hyena = AutoModel.from_pretrained(model_name, trust_remote_code=True)
        self.head = nn.Linear(self.hyena.config.d_model, 1) 
        self.clf = nn.Linear(self.hyena.config.d_model, num_labels)
    def forward(self, input_ids, attention_mask=None, labels=None):
        out = self.hyena(input_ids).last_hidden_state
        scores = self.head(out)
        if attention_mask is not None: scores = scores.masked_fill(attention_mask.unsqueeze(-1) == 0, float('-inf'))
        attn_probs = torch.softmax(scores, dim=1)
        pooled = torch.sum(out * attn_probs, dim=1)
        logits = self.clf(pooled)
        loss = nn.CrossEntropyLoss()(logits, labels) if labels is not None else None
        bp_index = torch.argmax(attn_probs, dim=1).squeeze(-1)
        return {'loss': loss, 'logits': logits, 'breakpoint_index': bp_index}

In [ ]:
# ==============================================================================
# CELL 3: METRICS & VISUALIZATION LOGIC (Restored Precision & Recall)
# ==============================================================================
from sklearn.metrics import precision_score, recall_score

def compute_metrics(y_true, y_probs):
    y_pred = np.argmax(y_probs, axis=1)
    
    # --- Standard Metrics ---
    acc = accuracy_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    
    # --- AUC Metrics ---
    try: 
        auroc = roc_auc_score(y_true, y_probs[:, 1])
    except ValueError: 
        auroc = 0.0
        
    return {
        "acc": acc, 
        "mcc": mcc, 
        "f1": f1, 
        "prec": prec, 
        "rec": rec, 
        "auroc": auroc
    }

def plot_comprehensive_results(y_true, y_probs, predicted_bps, title_prefix=""):
    y_pred = np.argmax(y_probs, axis=1)
    y_scores = y_probs[:, 1]
    
    plt.figure(figsize=(14, 10)) 
    
    # A. Confusion Matrix
    plt.subplot(2, 2, 1)
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Neg', 'Pos'], yticklabels=['Neg', 'Pos'])
    plt.title(f'{title_prefix} Confusion Matrix')

    # B. ROC Curve
    plt.subplot(2, 2, 2)
    fpr, tpr, _ = roc_curve(y_true, y_scores)
    plt.plot(fpr, tpr, label=f'AUC = {roc_auc_score(y_true, y_scores):.3f}', color='purple', lw=2)
    plt.plot([0, 1], [0, 1], 'k--', lw=1)
    plt.legend()
    plt.title(f'{title_prefix} ROC Curve')

    # C. Probability Distribution
    plt.subplot(2, 2, 3)
    neg_scores = y_scores[y_true == 0]
    pos_scores = y_scores[y_true == 1]
    sns.histplot(neg_scores, color='red', alpha=0.5, label='True Neg', bins=30, kde=True)
    sns.histplot(pos_scores, color='green', alpha=0.5, label='True Pos', bins=30, kde=True)
    plt.axvline(0.5, color='black', linestyle='--')
    plt.legend()
    plt.title(f'{title_prefix} Probability Dist.')

    # D. Genomic Index
    plt.subplot(2, 2, 4)
    high_conf = np.where((y_pred == 1) & (y_scores > Config.CONFIDENCE_THRESHOLD))[0]
    if len(high_conf) > 0:
        sns.histplot(predicted_bps[high_conf], kde=True, bins=50, color='blue', label='Predicted')
        plt.title(f'{title_prefix} Breakpoints (Conf > {Config.CONFIDENCE_THRESHOLD})')
        plt.xlabel('Genomic Index')
    else:
        plt.text(0.5, 0.5, "No High-Confidence Fusions", ha='center')
        plt.title('Breakpoint Locations')

    plt.tight_layout()
    plt.show()

In [ ]:
# ==============================================================================
# CELL 4: TRAINING & VALIDATION LOOPS
# ==============================================================================

def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    all_probs, all_labels = [], []
    for batch in tqdm(loader, desc="Training"):
        ids, mask, lbls = batch['input_ids'].to(device), batch['attention_mask'].to(device), batch['labels'].to(device)
        optimizer.zero_grad()
        outputs = model(ids, attention_mask=mask, labels=lbls)
        loss = outputs['loss']
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        probs = torch.softmax(outputs['logits'], dim=1)
        all_probs.extend(probs.detach().cpu().numpy())
        all_labels.extend(lbls.detach().cpu().numpy())
    metrics = compute_metrics(np.array(all_labels), np.array(all_probs))
    metrics['loss'] = total_loss / len(loader)
    return metrics

def validate(model, loader, device, desc="Validating"):
    model.eval()
    all_probs, all_labels, all_bps = [], [], []
    val_loss = 0
    with torch.no_grad():
        for batch in tqdm(loader, desc=desc):
            ids, mask, lbls = batch['input_ids'].to(device), batch['attention_mask'].to(device), batch['labels'].to(device)
            outputs = model(ids, attention_mask=mask, labels=lbls)
            val_loss += outputs['loss'].item()
            probs = torch.softmax(outputs['logits'], dim=1)
            bps = outputs['breakpoint_index']
            all_probs.extend(probs.detach().cpu().numpy())
            all_labels.extend(lbls.detach().cpu().numpy())
            all_bps.extend(bps.detach().cpu().numpy())
    metrics = compute_metrics(np.array(all_labels), np.array(all_probs))
    metrics['loss'] = val_loss / len(loader)
    return metrics, np.array(all_labels), np.array(all_probs), np.array(all_bps)

In [ ]:
# ==============================================================================
# CELL 5: LOAD ALL DATA & PERFORM 3-WAY SPLIT
# ==============================================================================
print("\n--- 1. LOADING ALL DATA ---")
preparator = DataPreparator(Config)

# A. Load All Positives
if os.path.exists(Config.POSITIVE_CSV):
    pos_df = pd.read_csv(Config.POSITIVE_CSV, header=None, names=DataPreparator.COLUMNS, sep='\t')
    pos_df['sequence'] = pos_df.apply(lambda r: preparator._apply_jitter(r["5'-gene sequence (10Kb)"], r["3'-gene sequence (10Kb)"]), axis=1)
    pos_df['label'] = 1
else:
    pos_df = pd.DataFrame({'sequence': [], 'label': []})

# Add Extra FASTA Positives
extra_seqs = []
for p in Config.EXTRA_POS_FASTA.split(','):
    extra_seqs.extend(preparator._load_fasta(p.strip()))
if extra_seqs:
    pos_df = pd.concat([pos_df[['sequence', 'label']], pd.DataFrame({'sequence': extra_seqs, 'label': 1})])

# B. Load All Negatives
neg_seqs = []
if os.path.exists(Config.NEGATIVE_CSV):
    neg_df = pd.read_csv(Config.NEGATIVE_CSV, header=None, names=DataPreparator.COLUMNS, sep='\t')
    neg_seqs.extend(neg_df.apply(lambda r: preparator._apply_jitter(r["5'-gene sequence (10Kb)"], r["3'-gene sequence (10Kb)"]), axis=1).tolist())

neg_seqs.extend(preparator._load_fasta(Config.NEG_FASTA_CANONICAL))
neg_seqs.extend(preparator._load_fasta(Config.NEG_FASTA_SYNTHETIC))
neg_df = pd.DataFrame({'sequence': neg_seqs, 'label': 0})

# Augmentation (Reverse Complement)
print("  > Augmenting Negatives (Reverse Complement)...")
aug_df = neg_df.copy()
aug_df['sequence'] = aug_df['sequence'].apply(preparator._get_reverse_complement)
neg_df = pd.concat([neg_df, aug_df], ignore_index=True)

# C. Combine & Dedup
full_df = pd.concat([pos_df[['sequence', 'label']], neg_df[['sequence', 'label']]], ignore_index=True)
full_df.drop_duplicates(subset=['sequence'], inplace=True)
full_df = full_df.sample(frac=1, random_state=Config.SEED).reset_index(drop=True)

print(f"✅ TOTAL UNIQUE POOL: {len(full_df)} samples")

# --- 2. PERFORM 3-WAY SPLIT (70% Train, 15% Val, 15% Test) ---
# First, split off Test (15%)
train_val_df, test_df = train_test_split(full_df, test_size=0.15, stratify=full_df['label'], random_state=Config.SEED)

# Then split remaining 85% into Train (approx 70% total) and Val (approx 15% total)
# 0.176 of 85% is ~15% of total
train_df, val_df = train_test_split(train_val_df, test_size=0.1765, stratify=train_val_df['label'], random_state=Config.SEED)

print(f"\n--- SPLIT STATISTICS ---")
print(f"TRAIN Set: {len(train_df)} (Pos: {sum(train_df['label'])})")
print(f"VAL   Set: {len(val_df)} (Pos: {sum(val_df['label'])})")
print(f"TEST  Set: {len(test_df)} (Pos: {sum(test_df['label'])})")

In [ ]:
# ==============================================================================
# CELL 6: INITIALIZE & TRAIN MODEL (With Safetensors Saving)
# ==============================================================================
from safetensors.torch import save_file 

# 1. ensure labels are available
train_seqs_list = train_df['sequence'].tolist()
train_lbls_list = train_df['label'].tolist()
val_seqs_list = val_df['sequence'].tolist()
val_lbls_list = val_df['label'].tolist()

# 2. Print Stats
n_train_pos = sum(train_lbls_list)
n_train_neg = len(train_lbls_list) - n_train_pos
n_val_pos = sum(val_lbls_list)
n_val_neg = len(val_lbls_list) - n_val_pos

print(f"--- DATASET STATISTICS ---")
print(f"Training Set:   {len(train_lbls_list)} samples")
print(f"  ├── Positives: {n_train_pos} ({n_train_pos/len(train_lbls_list):.1%})")
print(f"  └── Negatives: {n_train_neg} ({n_train_neg/len(train_lbls_list):.1%})")
print(f"Validation Set: {len(val_lbls_list)} samples")
print(f"  ├── Positives: {n_val_pos} ({n_val_pos/len(val_lbls_list):.1%})")
print(f"  └── Negatives: {n_val_neg} ({n_val_neg/len(val_lbls_list):.1%})")

# 3. Init Model
print(f"\nInit Tokenizer & Model: {Config.MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(Config.MODEL_NAME, trust_remote_code=True)
model = HyenaDNAClassifier(Config.MODEL_NAME).to(device)

# 4. Loaders
train_loader = DataLoader(
    DNABreakpointDataset(train_seqs_list, train_lbls_list, tokenizer, Config.MAX_LEN), 
    batch_size=Config.BATCH_SIZE, shuffle=True
)
val_loader = DataLoader(
    DNABreakpointDataset(val_seqs_list, val_lbls_list, tokenizer, Config.MAX_LEN), 
    batch_size=Config.BATCH_SIZE, shuffle=False
)

# 5. Optimizer
optimizer = AdamW(model.parameters(), lr=Config.LEARNING_RATE)
scheduler = get_linear_schedule_with_warmup(optimizer, 0, len(train_loader)*Config.EPOCHS)

# 6. Training Loop
print(f"\n Training Start ({Config.EPOCHS} Epochs)...")
for epoch in range(Config.EPOCHS):
    print(f"\n--- Epoch {epoch+1} ---")
    
    # Train
    train_metrics = train_epoch(model, train_loader, optimizer, scheduler, device)
    print(f"TRAIN >> Loss: {train_metrics['loss']:.4f} | Acc: {train_metrics['acc']:.4f} | F1: {train_metrics['f1']:.4f} | AUC: {train_metrics['auroc']:.4f}")
    
    # Validate
    val_metrics, y_true, y_probs, y_bps = validate(model, val_loader, device)
    print(f"VAL   >> Loss: {val_metrics['loss']:.4f} | Acc: {val_metrics['acc']:.4f} | F1: {val_metrics['f1']:.4f} | AUC: {val_metrics['auroc']:.4f}")
    
    # Save Epoch Checkpoint (Standard PyTorch)
    torch.save(model.state_dict(), f"{Config.OUTPUT_DIR}/model_epoch_{epoch+1}.pt")

print("\n--- Plotting VALIDATION Results ---")
plot_comprehensive_results(y_true, y_probs, y_bps, title_prefix="VAL")

# 7. SAVE FINAL MODEL (Safetensors Format)
print(f"\n Saving Final Model to: {Config.OUTPUT_DIR}/final_model")
final_dir = os.path.join(Config.OUTPUT_DIR, "final_model")
os.makedirs(final_dir, exist_ok=True)

# A. Save Weights (Safetensors)
save_file(model.state_dict(), os.path.join(final_dir, "model.safetensors"))

# B. Save Tokenizer
tokenizer.save_pretrained(final_dir)

# C. Save Config (Base Hyena Config)
model.hyena.config.save_pretrained(final_dir)

print("✅ Save Complete.")

In [ ]:
from datetime import datetime  # <--- Added this import
# ==============================================================================
# CELL 7: BLIND TEST SET EVALUATION
# ==============================================================================
print(f"{'='*40}")
print(f"PHASE 9: BLIND TEST SET EVALUATION")
print(f"{'='*40}")

# 1. VISUALIZATION FUNCTION (Defined locally to ensure it exists)
def plot_test_results_detailed(y_true, y_probs, predicted_bps):
    y_pred = np.argmax(y_probs, axis=1)
    y_scores = y_probs[:, 1] # Probability of being Positive (Fusion)
    
    plt.figure(figsize=(14, 10)) # Large 2x2 Grid
    
    # --- A. Confusion Matrix ---
    plt.subplot(2, 2, 1)
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Non-Fusion', 'Fusion'], 
                yticklabels=['Non-Fusion', 'Fusion'])
    plt.title('A. Confusion Matrix')

    # --- B. ROC Curve ---
    plt.subplot(2, 2, 2)
    fpr, tpr, _ = roc_curve(y_true, y_scores)
    plt.plot(fpr, tpr, label=f'AUC = {roc_auc_score(y_true, y_scores):.3f}', color='purple', lw=2)
    plt.plot([0, 1], [0, 1], 'k--', lw=1)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.legend()
    plt.title('B. ROC Curve')

    # --- C. Breakpoint Probability (Confidence Distribution) ---
    plt.subplot(2, 2, 3)
    # Separate scores for True Negatives and True Positives
    neg_scores = y_scores[y_true == 0]
    pos_scores = y_scores[y_true == 1]
    
    sns.histplot(neg_scores, color='red', alpha=0.5, label='True Negatives', bins=30, kde=True)
    sns.histplot(pos_scores, color='green', alpha=0.5, label='True Positives', bins=30, kde=True)
    plt.axvline(0.5, color='black', linestyle='--', label='Threshold')
    plt.xlabel('Model Probability (Fusion Confidence)')
    plt.legend()
    plt.title('C. Breakpoint Probability Distribution')

    # --- D. Genomic Index (Breakpoint Locations) ---
    plt.subplot(2, 2, 4)
    # Filter: Only show locations for High-Confidence Predictions (> 0.9)
    high_conf_indices = np.where((y_pred == 1) & (y_scores > Config.CONFIDENCE_THRESHOLD))[0]
    
    if len(high_conf_indices) > 0:
        valid_bps = predicted_bps[high_conf_indices]
        sns.histplot(valid_bps, kde=True, bins=50, color='blue', label='Predicted Breakpoints')
        plt.xlabel('Genomic Index (Sequence Length 0-20480)')
        plt.title(f'D. Genomic Index (High-Conf > {Config.CONFIDENCE_THRESHOLD})')
    else:
        plt.text(0.5, 0.5, "No High-Confidence Fusions Found", ha='center')
        plt.title('D. Genomic Index (No Data)')

    plt.tight_layout()
    plt.show()

# 2. STATISTICS
# Use the pre-split Test DataFrame (test_df) created in Cell 5
test_labels = test_df['label'].tolist()
n_test_pos = sum(test_labels)
n_test_neg = len(test_labels) - n_test_pos

print(f"--- TEST SET STATISTICS ---")
print(f"Total Samples:   {len(test_labels)}")
print(f"  ├── Positives: {n_test_pos} ({n_test_pos/len(test_labels):.1%})")
print(f"  └── Negatives: {n_test_neg} ({n_test_neg/len(test_labels):.1%})")

# 3. PREPARE LOADER
# Note: shuffling is False for testing to ensure labels align
test_loader = DataLoader(
    DNABreakpointDataset(test_df['sequence'].tolist(), test_labels, tokenizer, Config.MAX_LEN), 
    batch_size=Config.BATCH_SIZE, 
    shuffle=False
)

# 4. RUN VALIDATION LOGIC
test_metrics, t_true, t_probs, t_bps = validate(model, test_loader, device, desc="Testing")

# 5. PRINT RESULTS
print(f"\n🏆 FINAL TEST RESULTS:")
print(f"   Accuracy:  {test_metrics['acc']:.4f}")
print(f"   Precision: {test_metrics['prec']:.4f}")
print(f"   Recall:    {test_metrics['rec']:.4f}")
print(f"   F1 Score:  {test_metrics['f1']:.4f}")
print(f"   AUC:       {test_metrics['auroc']:.4f}")
print(f"   MCC:       {test_metrics['mcc']:.4f}")

# 6. VISUALIZE
plot_test_results_detailed(t_true, t_probs, t_bps)

# 7. SAVE RESULTS TO CSV (NEW SECTION)
# ==============================================================================
# Generate a timestamped filename
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_filename = f"blind_test_predictions_{timestamp}.csv"

print(f"\n EXPORTING RESULTS...")

# Create a DataFrame using the original test_df to preserve metadata (like IDs/Sequences)
results_df = test_df.copy()

# Calculate class predictions from probabilities
# t_probs is typically [N, 2], we take the index of the max value
pred_classes = np.argmax(t_probs, axis=1)
pred_scores_fusion = t_probs[:, 1]

# Append Model Outputs to the DataFrame
results_df['True_Label'] = t_true
results_df['Predicted_Label'] = pred_classes
results_df['Fusion_Prob'] = pred_scores_fusion
results_df['Predicted_Breakpoint'] = t_bps

# Add a 'Result_Type' column for easy Excel filtering (TP, FP, TN, FN)
results_df['Result_Type'] = results_df.apply(
    lambda x: 'TP' if (x['True_Label'] == 1 and x['Predicted_Label'] == 1) else
              ('TN' if (x['True_Label'] == 0 and x['Predicted_Label'] == 0) else
              ('FP' if (x['True_Label'] == 0 and x['Predicted_Label'] == 1) else 'FN')),
    axis=1
)

# Save to CSV
results_df.to_csv(csv_filename, index=False)
print(f" Predictions successfully saved to: {csv_filename}")
print(f"   (Contains columns: True_Label, Predicted_Label, Fusion_Prob, Result_Type)")